In [4]:
import bisect
import heapq
from typing import List, Dict, Tuple, Optional, Any
from dataclasses import dataclass, field
from collections import defaultdict, deque
import time
from functools import total_ordering

@total_ordering
@dataclass
class Order:
    """Order class with total ordering for priority queue"""
    # For bids: higher price first, then earlier timestamp
    # For asks: lower price first, then earlier timestamp
    timestamp: float
    order_id: str
    side: str  # 'B' for bid, 'A' for ask
    price: float
    qty: int
    remaining_qty: int = field(init=False)

    def __post_init__(self):
        self.remaining_qty = self.qty
        # Create a priority key based on side
        if self.side == 'B':
            # For bids: negative price for max-heap behavior with heapq (higher bids first)
            self._priority_price = -self.price
        else:
            # For asks: positive price (lower asks come first)
            self._priority_price = self.price

        # Store priority key tuple for comparisons
        self._priority_key = (self._priority_price, self.timestamp)

    def __eq__(self, other):
        """Equality comparison based on priority key"""
        if not isinstance(other, Order):
            return NotImplemented
        return self._priority_key == other._priority_key

    def __lt__(self, other):
        """Less-than comparison for heapq based on priority_key"""
        if not isinstance(other, Order):
            return NotImplemented
        return self._priority_key < other._priority_key

@dataclass
class Trade:
    """Trade execution record"""
    timestamp: float
    buy_order_id: str
    sell_order_id: str
    price: float
    qty: int

class OrderBook:
    def __init__(self):
        # Priority queues for bids and asks
        self.bids: List[Order] = []  # Max-heap for bids (higher prices first)
        self.asks: List[Order] = []  # Min-heap for asks (lower prices first)

        # Dictionary for quick order lookup by ID
        self.orders: Dict[str, Order] = {}

        # Track trades
        self.trades: List[Trade] = []

        # Price levels tracking for depth
        self.bid_levels: Dict[float, int] = defaultdict(int)
        self.ask_levels: Dict[float, int] = defaultdict(int)

        # Order IDs at price levels for quick removal
        self.orders_at_price: Dict[Tuple[str, float], List[str]] = defaultdict(list)

    def _add_to_price_levels(self, order: Order):
        """Add order to price level tracking"""
        if order.side == 'B':
            self.bid_levels[order.price] += order.remaining_qty
        else:
            self.ask_levels[order.price] += order.remaining_qty

        # Track order ID at this price level
        price_key = (order.side, order.price)
        self.orders_at_price[price_key].append(order.order_id)

    def _remove_from_price_levels(self, order: Order, qty: int):
        """Remove quantity from price level tracking"""
        if order.side == 'B':
            self.bid_levels[order.price] -= qty
            if self.bid_levels[order.price] <= 0:
                del self.bid_levels[order.price]
        else:
            self.ask_levels[order.price] -= qty
            if self.ask_levels[order.price] <= 0:
                del self.ask_levels[order.price]

        # Remove order ID from price level tracking if fully filled
        if order.remaining_qty == 0:
            price_key = (order.side, order.price)
            if order.order_id in self.orders_at_price[price_key]:
                self.orders_at_price[price_key].remove(order.order_id)

    def _cleanup_empty_orders(self, heap: List[Order], is_bid: bool):
        """Remove fully filled orders from the heap"""
        while heap and heap[0].remaining_qty == 0:
            order = heapq.heappop(heap)
            if order.order_id in self.orders:
                # Remove from orders dict
                del self.orders[order.order_id]

                # Remove from price level tracking if not already done
                price_key = (order.side, order.price)
                if order.order_id in self.orders_at_price[price_key]:
                    self.orders_at_price[price_key].remove(order.order_id)

    def add_order(self, order: Order) -> List[Trade]:
        """Add order to book and execute trades if possible"""
        trades = []

        if order.side == 'B':
            # Try to match with asks
            while self.asks and order.remaining_qty > 0:
                best_ask = self.asks[0]

                # Check if bid price >= ask price (crossing the spread)
                if order.price >= best_ask.price:
                    # Execute trade
                    trade_qty = min(order.remaining_qty, best_ask.remaining_qty)
                    execution_price = best_ask.price  # Price is the ask price

                    # Record trade
                    trade = Trade(
                        timestamp=time.time(),
                        buy_order_id=order.order_id,
                        sell_order_id=best_ask.order_id,
                        price=execution_price,
                        qty=trade_qty
                    )
                    trades.append(trade)
                    self.trades.append(trade)

                    # Update quantities
                    order.remaining_qty -= trade_qty
                    best_ask.remaining_qty -= trade_qty

                    # Update price levels
                    self._remove_from_price_levels(best_ask, trade_qty)

                    # Remove ask if fully filled
                    if best_ask.remaining_qty == 0:
                        heapq.heappop(self.asks)
                        if best_ask.order_id in self.orders:
                            del self.orders[best_ask.order_id]
                else:
                    # No more crossing orders
                    break

            # Clean up any fully filled asks at the top
            self._cleanup_empty_orders(self.asks, False)

            # Add remaining bid to book
            if order.remaining_qty > 0:
                heapq.heappush(self.bids, order)
                self.orders[order.order_id] = order

                # Update bid levels
                self._add_to_price_levels(order)

        else:  # order.side == 'A'
            # Try to match with bids
            while self.bids and order.remaining_qty > 0:
                best_bid = self.bids[0]

                # Check if ask price <= bid price (crossing the spread)
                if order.price <= best_bid.price:
                    # Execute trade
                    trade_qty = min(order.remaining_qty, best_bid.remaining_qty)
                    execution_price = best_bid.price  # Price is the bid price

                    # Record trade
                    trade = Trade(
                        timestamp=time.time(),
                        buy_order_id=best_bid.order_id,
                        sell_order_id=order.order_id,
                        price=execution_price,
                        qty=trade_qty
                    )
                    trades.append(trade)
                    self.trades.append(trade)

                    # Update quantities
                    order.remaining_qty -= trade_qty
                    best_bid.remaining_qty -= trade_qty

                    # Update price levels
                    self._remove_from_price_levels(best_bid, trade_qty)

                    # Remove bid if fully filled
                    if best_bid.remaining_qty == 0:
                        heapq.heappop(self.bids)
                        if best_bid.order_id in self.orders:
                            del self.orders[best_bid.order_id]
                else:
                    # No more crossing orders
                    break

            # Clean up any fully filled bids at the top
            self._cleanup_empty_orders(self.bids, True)

            # Add remaining ask to book
            if order.remaining_qty > 0:
                heapq.heappush(self.asks, order)
                self.orders[order.order_id] = order

                # Update ask levels
                self._add_to_price_levels(order)

        return trades

    def cancel_order(self, order_id: str) -> bool:
        """Cancel an existing order"""
        if order_id not in self.orders:
            return False

        order = self.orders[order_id]

        # Remove from price levels
        self._remove_from_price_levels(order, order.remaining_qty)

        # Mark order as cancelled (set remaining_qty to 0)
        order.remaining_qty = 0

        # Remove from orders dictionary
        del self.orders[order_id]

        # Clean up the heap (lazy removal)
        if order.side == 'B':
            self._cleanup_empty_orders(self.bids, True)
        else:
            self._cleanup_empty_orders(self.asks, False)

        return True

    def get_top_levels(self, levels: int = 5) -> Dict[str, List[Tuple[float, int]]]:
        """Get top N price levels for bids and asks"""
        result = {'bids': [], 'asks': []}

        # Get top bid levels (sorted descending by price)
        sorted_bids = sorted(self.bid_levels.items(), key=lambda x: x[0], reverse=True)
        result['bids'] = [(price, qty) for price, qty in sorted_bids[:levels] if qty > 0]

        # Get top ask levels (sorted ascending by price)
        sorted_asks = sorted(self.ask_levels.items(), key=lambda x: x[0])
        result['asks'] = [(price, qty) for price, qty in sorted_asks[:levels] if qty > 0]

        return result

    def print_depth(self, levels: int = 5):
        """Print current market depth"""
        top_levels = self.get_top_levels(levels)

        print("\n" + "="*60)
        print(f"MARKET DEPTH (Top {levels} levels)")
        print("="*60)

        # Print asks (descending - highest ask first)
        print("\nASKS (Sell):")
        print(f"{'Price':<12} {'Quantity':<12}")
        print("-"*24)
        if top_levels['asks']:
            for price, qty in reversed(top_levels['asks']):  # Highest ask first
                print(f"{price:<12.2f} {qty:<12}")
        else:
            print("No asks")

        # Print spread if available
        if top_levels['bids'] and top_levels['asks']:
            best_bid = top_levels['bids'][0][0]
            best_ask = top_levels['asks'][0][0]
            spread = best_ask - best_bid
            print(f"\nSpread: {spread:.4f} (Best Bid: {best_bid:.2f} | Best Ask: {best_ask:.2f})")
        elif top_levels['bids']:
            print(f"\nBest Bid: {top_levels['bids'][0][0]:.2f} | No asks")
        elif top_levels['asks']:
            print(f"\nNo bids | Best Ask: {top_levels['asks'][0][0]:.2f}")
        else:
            print("\nEmpty order book")

        # Print bids (descending - highest bid first)
        print("\nBIDS (Buy):")
        print(f"{'Price':<12} {'Quantity':<12}")
        print("-"*24)
        if top_levels['bids']:
            for price, qty in top_levels['bids']:  # Already sorted highest first
                print(f"{price:<12.2f} {qty:<12}")
        else:
            print("No bids")

        print("="*60 + "\n")

    def get_trades(self) -> List[Trade]:
        """Get all executed trades"""
        return self.trades.copy()

    def clear(self):
        """Clear the order book"""
        self.bids.clear()
        self.asks.clear()
        self.orders.clear()
        self.trades.clear()
        self.bid_levels.clear()
        self.ask_levels.clear()
        self.orders_at_price.clear()

def process_order_stream(order_book: OrderBook, orders: List[Tuple[str, str, float, int]]) -> List[Trade]:
    """Process a stream of orders and return trades"""
    all_trades = []
    current_time = time.time()

    for i, (side, order_id, price, qty) in enumerate(orders):
        timestamp = current_time + i * 0.001  # Add small increment for ordering
        order = Order(
            timestamp=timestamp,
            order_id=order_id,
            side=side,
            price=price,
            qty=qty
        )

        trades = order_book.add_order(order)
        all_trades.extend(trades)

    return all_trades

def print_trades(trades: List[Trade]):
    """Print trade execution details"""
    if not trades:
        print("\nNo trades executed")
        return

    print("\n" + "="*80)
    print("TRADE EXECUTIONS")
    print("="*80)
    print(f"{'Timestamp':<12} {'Buy Order':<12} {'Sell Order':<12} {'Price':<10} {'Qty':<8}")
    print("-"*80)

    for trade in trades:
        time_str = f"{trade.timestamp:.6f}"[-10:]
        print(f"{time_str:<12} {trade.buy_order_id:<12} {trade.sell_order_id:<12} "
              f"{trade.price:<10.2f} {trade.qty:<8}")

    print(f"\nTotal Trades: {len(trades)}")
    total_qty = sum(trade.qty for trade in trades)
    print(f"Total Volume: {total_qty}")
    print("="*80)

def verify_identical_trades(trades1: List[Trade], trades2: List[Trade]) -> bool:
    """Verify that two trade lists are identical"""
    if len(trades1) != len(trades2):
        print(f"✗ FAIL: Different number of trades: {len(trades1)} vs {len(trades2)}")
        return False

    for i, (t1, t2) in enumerate(zip(trades1, trades2)):
        if (t1.buy_order_id != t2.buy_order_id or
            t1.sell_order_id != t2.sell_order_id or
            abs(t1.price - t2.price) > 0.0001 or
            t1.qty != t2.qty):
            print(f"✗ FAIL: Trade {i} mismatch:")
            print(f"  Trade1: Buy={t1.buy_order_id}, Sell={t1.sell_order_id}, "
                  f"Price={t1.price:.4f}, Qty={t1.qty}")
            print(f"  Trade2: Buy={t2.buy_order_id}, Sell={t2.sell_order_id}, "
                  f"Price={t2.price:.4f}, Qty={t2.qty}")
            return False

    print(f"✓ PASS: All {len(trades1)} trades are identical")
    return True

def main():
    """Main demonstration function"""
    # Example order stream - more comprehensive test cases
    orders = [
        # (side, order_id, price, qty)
        ('B', 'B1', 100.00, 100),  # Buy 100 @ 100.00
        ('B', 'B2', 99.50, 50),    # Buy 50 @ 99.50
        ('B', 'B3', 100.50, 75),   # Buy 75 @ 100.50 (new best bid)
        ('A', 'A1', 101.00, 80),   # Sell 80 @ 101.00
        ('A', 'A2', 100.25, 120),  # Sell 120 @ 100.25 (crosses with B3)
        ('B', 'B4', 100.75, 60),   # Buy 60 @ 100.75 (crosses with A2)
        ('A', 'A3', 99.00, 40),    # Sell 40 @ 99.00 (crosses with B1, B2)
        ('B', 'B5', 101.00, 30),   # Buy 30 @ 101.00 (crosses with A2)
        ('A', 'A4', 98.50, 100),   # Sell 100 @ 98.50 (crosses with all bids)
    ]

    print("="*80)
    print("LIMIT ORDER BOOK - DEMONSTRATION")
    print("="*80)
    print("\nOrder Stream to Process:")
    print("Side\tOrderID\tPrice\t\tQty")
    print("-"*40)
    for order in orders:
        print(f"{order[0]}\t{order[1]}\t{order[2]:.2f}\t\t{order[3]}")

    # First pass
    print("\n" + "="*80)
    print("FIRST PASS - Processing Order Stream")
    print("="*80)

    order_book1 = OrderBook()
    trades1 = process_order_stream(order_book1, orders)

    # Print market depth after first pass
    order_book1.print_depth(5)

    # Print trades from first pass
    print_trades(trades1)

    # Second pass (with fresh order book)
    print("\n" + "="*80)
    print("SECOND PASS - Reprocessing Order Stream")
    print("="*80)

    order_book2 = OrderBook()
    trades2 = process_order_stream(order_book2, orders)

    # Print market depth after second pass
    order_book2.print_depth(5)

    # Print trades from second pass
    print_trades(trades2)

    # Verify identical trades
    print("\n" + "="*80)
    print("VERIFICATION: Comparing Trade Execution Results")
    print("="*80)

    identical_trades = verify_identical_trades(trades1, trades2)

    # Verify order book state
    depth1 = order_book1.get_top_levels(5)
    depth2 = order_book2.get_top_levels(5)

    if depth1 == depth2:
        print("✓ PASS: Order book depth is identical in both passes")
    else:
        print("✗ FAIL: Order book depth differs between passes")
        print(f"  First pass depth:\n  Bids: {depth1['bids']}\n  Asks: {depth1['asks']}")
        print(f"  Second pass depth:\n  Bids: {depth2['bids']}\n  Asks: {depth2['asks']}")

    # Additional demonstration
    print("\n" + "="*80)
    print("ADDITIONAL OPERATIONS DEMONSTRATION")
    print("="*80)

    # Add some more orders
    print("\nAdding additional orders to order book from first pass...")
    additional_orders = [
        ('B', 'B6', 99.75, 25),
        ('A', 'A5', 100.50, 15),
        ('B', 'B7', 101.25, 10),
    ]

    print("\nAdditional Orders:")
    print("Side\tOrderID\tPrice\t\tQty")
    print("-"*40)
    for order in additional_orders:
        print(f"{order[0]}\t{order[1]}\t{order[2]:.2f}\t\t{order[3]}")

    for side, order_id, price, qty in additional_orders:
        order = Order(
            timestamp=time.time(),
            order_id=order_id,
            side=side,
            price=price,
            qty=qty
        )
        trades = order_book1.add_order(order)
        if trades:
            print(f"\nTrade(s) executed from {order_id}:")
            for trade in trades:
                print(f"  {trade.buy_order_id} buys {trade.qty} from "
                      f"{trade.sell_order_id} @ {trade.price:.2f}")

    print("\nUpdated order book depth:")
    order_book1.print_depth(5)

    # Demonstrate order cancellation
    print("\nCancelling order B2 (if exists)...")
    if order_book1.cancel_order('B2'):
        print("✓ Order B2 cancelled successfully")
    else:
        print("✗ Order B2 not found (may have been filled)")

    print("\nOrder book depth after cancellation:")
    order_book1.print_depth(5)

    # Show statistics
    print("\n" + "="*80)
    print("ORDER BOOK STATISTICS")
    print("="*80)
    print(f"Total orders processed: {len(order_book1.orders)}")
    print(f"Total trades executed: {len(order_book1.trades)}")
    total_volume = sum(trade.qty for trade in order_book1.trades)
    print(f"Total trade volume: {total_volume}")

    if order_book1.trades:
        prices = [trade.price for trade in order_book1.trades]
        print(f"Average trade price: {sum(prices)/len(prices):.4f}")
        print(f"Price range: {min(prices):.4f} - {max(prices):.4f}")

    depth = order_book1.get_top_levels(5)
    total_bid_qty = sum(qty for _, qty in depth['bids'])
    total_ask_qty = sum(qty for _, qty in depth['asks'])
    print(f"Total bid quantity (top 5 levels): {total_bid_qty}")
    print(f"Total ask quantity (top 5 levels): {total_ask_qty}")

def test_edge_cases():
    """Test edge cases and corner scenarios"""
    print("\n" + "="*80)
    print("EDGE CASE TESTING")
    print("="*80)

    # Test 1: Exact price match
    print("\nTest 1: Exact price matching")
    book = OrderBook()

    # Add a bid
    bid = Order(time.time(), 'B1', 'B', 100.00, 100)
    book.add_order(bid)

    # Add matching ask
    ask = Order(time.time() + 0.001, 'A1', 'A', 100.00, 50)
    trades = book.add_order(ask)

    print(f"  Bid @ 100.00, Ask @ 100.00 -> {len(trades)} trade(s) executed")
    if trades:
        print(f"  Trade: {trades[0].qty} @ {trades[0].price}")

    # Test 2: Partial fill
    print("\nTest 2: Partial order fill")
    book = OrderBook()

    # Add a large bid
    bid1 = Order(time.time(), 'B1', 'B', 100.00, 200)
    book.add_order(bid1)

    # Add smaller ask (partial fill)
    ask1 = Order(time.time() + 0.001, 'A1', 'A', 100.00, 50)
    trades = book.add_order(ask1)

    print(f"  Bid 200 @ 100.00, Ask 50 @ 100.00 -> {len(trades)} trade(s)")
    print(f"  Remaining bid quantity: {book.orders['B1'].remaining_qty}")

    # Test 3: Multiple orders at same price
    print("\nTest 3: Multiple orders at same price (time priority)")
    book = OrderBook()

    # Add multiple bids at same price
    for i in range(3):
        bid = Order(time.time() + i * 0.001, f'B{i+1}', 'B', 100.00, 100)
        book.add_order(bid)

    # Add ask that matches all
    ask = Order(time.time() + 0.01, 'A1', 'A', 100.00, 250)
    trades = book.add_order(ask)

    print(f"  3 bids @ 100.00 (100 each), Ask 250 @ 100.00")
    print(f"  -> {len(trades)} trade(s), total filled: {sum(t.qty for t in trades)}")

    # Check which orders were filled
    filled_orders = set()
    for trade in trades:
        filled_orders.add(trade.buy_order_id)

    print(f"  Orders filled (by time priority): {sorted(filled_orders)}")

if __name__ == "__main__":
    main()
    test_edge_cases()

LIMIT ORDER BOOK - DEMONSTRATION

Order Stream to Process:
Side	OrderID	Price		Qty
----------------------------------------
B	B1	100.00		100
B	B2	99.50		50
B	B3	100.50		75
A	A1	101.00		80
A	A2	100.25		120
B	B4	100.75		60
A	A3	99.00		40
B	B5	101.00		30
A	A4	98.50		100

FIRST PASS - Processing Order Stream

MARKET DEPTH (Top 5 levels)

ASKS (Sell):
Price        Quantity    
------------------------
101.00       50          

Spread: 1.5000 (Best Bid: 99.50 | Best Ask: 101.00)

BIDS (Buy):
Price        Quantity    
------------------------
99.50        25          


TRADE EXECUTIONS
Timestamp    Buy Order    Sell Order   Price      Qty     
--------------------------------------------------------------------------------
259.947054   B3           A2           100.50     75      
259.947068   B4           A2           100.25     45      
259.947078   B4           A3           100.75     15      
259.947083   B1           A3           100.00     25      
259.947088   B5           A1        